# 32 — Junior authors' self-citations to award paper: descriptive analysis

This notebook takes the author-level summary from `31_junior_self_cites_awardpaper.ipynb` and explores how junior authors' self-citation share to their award paper varies across conferences and award years.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

sns.set_theme(style='whitegrid')

BASE = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories')
DATA_DERIVED = BASE / 'data' / 'derived'

PATH_SELF = DATA_DERIVED / 'junior_awardpaper_self_cites_summary.csv'

df = pd.read_csv(PATH_SELF)
print('Loaded self-citation summary:', df.shape)
df.head()

In [ ]:
# Basic cleaning / sanity

df['award_conference'] = df['award_conference'].astype(str)
df['award_year'] = pd.to_numeric(df['award_year'], errors='coerce')

df['share_self_citing'] = df['share_self_citing'].fillna(0.0)
df['n_citing_papers_total_post_award'] = pd.to_numeric(
    df['n_citing_papers_total_post_award'], errors='coerce'
)
df = df[df['n_citing_papers_total_post_award'] > 0].copy()

print('After cleaning:', df.shape)
df.describe(include='all').T.head(12)

## 1. Overall distribution of junior self-citation share


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['share_self_citing'], bins=30, ax=axes[0])
axes[0].set_title('Junior self-citation share to award paper')
axes[0].set_xlabel('Share of post-award citing papers that include junior author')

sns.histplot(df['n_citing_papers_total_post_award'], bins=30, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Total post-award citing papers (log y)')
axes[1].set_xlabel('Number of post-award citing papers to award paper')

plt.tight_layout()
plt.show()

## 2. Junior self-citation share by conference


In [ ]:
conf_stats = (
    df.groupby('award_conference', as_index=False)
      .agg(
          n_junior_authors=('junior_author_id', 'count'),
          mean_share=('share_self_citing', 'mean'),
          median_share=('share_self_citing', 'median')
      )
      .sort_values('mean_share', ascending=False)
)

conf_stats.head(15)

In [ ]:
top_confs = df['award_conference'].value_counts().head(10).index
df_top = df[df['award_conference'].isin(top_confs)].copy()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_top, x='award_conference', y='share_self_citing')
plt.xticks(rotation=45)
plt.ylabel('Junior self-citation share')
plt.title('Distribution of junior self-citation share by conference (top 10)')
plt.tight_layout()
plt.show()

## 3. Junior self-citation share over award year


In [ ]:
year_stats = (
    df.groupby('award_year', as_index=False)
      .agg(
          mean_share=('share_self_citing', 'mean'),
          median_share=('share_self_citing', 'median'),
          n_junior_authors=('junior_author_id', 'count')
      )
)

year_stats.head()

In [ ]:
plt.figure(figsize=(8, 4))
sns.lineplot(data=year_stats, x='award_year', y='mean_share', marker='o')
plt.ylabel('Mean junior self-citation share')
plt.xlabel('Award year')
plt.title('Junior self-citation share over award years')
plt.tight_layout()
plt.show()

## 4. Relationship between self-citation share and size of citation impact


In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(
    data=df,
    x='n_citing_papers_total_post_award',
    y='share_self_citing',
    alpha=0.4
)
plt.xscale('log')
plt.xlabel('Total post-award citing papers (log scale)')
plt.ylabel('Junior self-citation share')
plt.title('Do highly cited award papers have different self-citation shares?')
plt.tight_layout()
plt.show()